# Water Quality Prediction using Deep Learning

This notebook aims to predict the **Water Quality Index (WQI)** and **Water Quality Classification** using deep learning neural networks based on the dataset provided by the Central Pollution Control Board (CPCB).

### Project Overview:
- **Target variable for Regression:** Water Quality Index (WQI)
- **Target variable for Classification:** Water Quality Classification
- **Features:** Chemical and physical parameters such as pH, EC, HCO3, Cl, SO4, NO3, etc.

---

## 1. Setup and Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, accuracy_score, f1_score, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## 2. Data Loading and Cleaning

In [ ]:
file_path = "Dataset/WknaT6YxR02YeXYuMDPg_water_quality.csv"

# Preprocess the CSV file to ignore the garbage text in the first line
with open(file_path, 'r', encoding='latin1') as f:
    first_line = f.readline()
    header_start = first_line.find("Well_ID")
    cleaned_header = first_line[header_start:].strip()

# Reading the actual data skipping the garbage first line
df = pd.read_csv(file_path, skiprows=1, header=None, encoding='latin1')
columns = cleaned_header.split(',')
df.columns = columns

print("Initial DataFrame Head:")
display(df.head())
print("Shape:", df.shape)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Drop non-essential columns for prediction
drop_cols = ['Well_ID', 'State', 'District', 'Block', 'Village']
data = df.drop(columns=drop_cols)

# Ensure all feature columns are numeric
feature_cols = ['pH', 'EC', 'CO3', 'HCO3', 'Cl', 'SO4', 'NO3', 'TH', 'Ca', 'Mg', 'Na', 'K', 'F', 'TDS', 'Latitude', 'Longitude', 'Year']
for col in feature_cols + ['WQI']:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# Summary of missing values
print("Missing values pre-cleaning:")
print(data.isnull().sum())

# Drop rows where targets are NaN
data = data.dropna(subset=['WQI', 'Water Quality Classification'])

# Fill feature NaNs with median
data[feature_cols] = data[feature_cols].apply(lambda x: x.fillna(x.median()))

print("
Post-cleaning shape:", data.shape)

In [ ]:
# Visualizing distributions of WQI
plt.figure(figsize=(10, 5))
sns.histplot(data['WQI'], bins=30, kde=True)
plt.title('WQI Distribution')
plt.show()

# Class distribution
plt.figure(figsize=(10, 5))
data['Water Quality Classification'].value_counts().plot(kind='bar')
plt.title('Classification Distribution')
plt.xticks(rotation=45)
plt.show()

## 4. Data Preprocessing and Splitting

In [ ]:
# Encoding the categorical target
le = LabelEncoder()
data['Class_Encoded'] = le.fit_transform(data['Water Quality Classification'].astype(str))

X = data[feature_cols]
y_reg = data['WQI']
y_cls = data['Class_Encoded']

# Train-test split for regression
X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
# Same split approach for classification to maintain consistency if needed (or separate splits)
_, _, y_cls_train, y_cls_test = train_test_split(X, y_cls, test_size=0.2, random_state=42)

# Scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data shape:", X_train.shape)
print("Number of classes:", len(le.classes_))
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

## 5. Regression Model for WQI

In [ ]:
def build_regression_model():
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)  # Linear output for regression
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

reg_model = build_regression_model()
print(reg_model.summary())

# Train with early stopping
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_reg = reg_model.fit(
    X_train_scaled, y_reg_train, 
    validation_split=0.1, 
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stop], 
    verbose=1
)

In [ ]:
# Regression Model Evaluation
y_reg_pred = reg_model.predict(X_test_scaled)
r2 = r2_score(y_reg_test, y_reg_pred)
print(f"R² Score: {r2:.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(y_reg_test, y_reg_pred, alpha=0.5)
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--')
plt.xlabel('Actual WQI')
plt.ylabel('Predicted WQI')
plt.title('Actual vs Predicted WQI')
plt.show()

## 6. Classification Model for Water Quality Classification

In [ ]:
def build_classification_model():
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(len(le.classes_), activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cls_model = build_classification_model()
print(cls_model.summary())

history_cls = cls_model.fit(
    X_train_scaled, y_cls_train, 
    validation_split=0.1, 
    epochs=100, 
    batch_size=32, 
    callbacks=[early_stop], 
    verbose=1
)

In [ ]:
# Classification Model Evaluation
y_cls_pred_probs = cls_model.predict(X_test_scaled)
y_cls_pred = np.argmax(y_cls_pred_probs, axis=1)

accuracy = accuracy_score(y_cls_test, y_cls_pred)
f1 = f1_score(y_cls_test, y_cls_pred, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score (Weighted): {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_cls_test, y_cls_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 7. Conclusions

In this project, neural network architectures were built for both regression and classification on the water quality dataset provided by the CPCB. 

Specific metrics achieved:
- **Regression (WQI):** R² Score shows how well the model explains variance in the Water Quality Index.
- **Classification (Water Quality Classification):** Accuracy and F1-score reveal the model's ability to categorize different quality levels effectively.

The data required significant cleaning due to some noise in the first line and missing values in specific parameters. Scaling all features ensured the neural networks could converge more efficiently.